# ÇÖZÜM: K-En Yakın Komşu (KNN) ile Cinsiyet Tahmini

Bu çalışma kitabı, KNN ile cinsiyet tahmini öğrenci görevindeki eksik kısımların tamamlanmış **çözüm (cevap anahtarı)** versiyonudur.

## 1. Gerekli Kütüphanelerin Yüklenmesi ve Verinin Hazırlanması

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

sns.set_theme(style="whitegrid")

# Veriyi yükleme
df = pd.read_excel('ayakkabi_numarasi.xlsx')

# Cinsiyet sütununu sayısal hedefe dönüştürme (Erkek: 1, Kadın: 0)
df['Cinsiyet_E'] = (df['Cinsiyet(E veya K)'] == 'E').astype(int)
df.head()

### ÇÖZÜM 1: X ve y Değişkenlerinin Ayrılması ve Eğitim/Test Bölünmesi

In [ ]:
# X için bağımsız değişken sütun isimleri
feature_cols = ['Kilo(kg)', 'Boy(metre)', 'Yaş', 'Ayakkabı Numarası']
X = df[feature_cols]

# y için hedef değişken sütun ismi
y = df['Cinsiyet_E']

# Veriyi %20 test olacak şekilde ayırma
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42
)

print(f"Eğitim seti boyutu: {X_train.shape[0]}, Test seti boyutu: {X_test.shape[0]}")

### ÇÖZÜM 2: Verilerin Ölçeklendirilmesi (StandardScaler)

In [ ]:
scaler = StandardScaler()

# Eğitim verisini öğrenip dönüştürme
X_train_scaled = scaler.fit_transform(X_train)

# Test verisini eğitimdeki ölçeğe göre dönüştürme
X_test_scaled = scaler.transform(X_test)

### ÇÖZÜM 3: KNN Modelinin Kurulması ve Eğitilmesi

In [ ]:
# Komşu sayısını (n_neighbors) 5 olarak ayarlama
model = KNeighborsClassifier(n_neighbors=5)

# Modeli ölçeklendirilmiş eğitim verisi ve y_train ile eğitme
model.fit(X_train_scaled, y_train)

print("KNN modeli başarıyla eğitildi!")

### ÇÖZÜM 4: Tahmin Yapma ve Modeli Değerlendirme

In [ ]:
# Ölçeklendirilmiş test verisi ile tahmin üretme
y_pred = model.predict(X_test_scaled)

# Doğruluk (Accuracy) skorunu hesaplama
acc = accuracy_score(y_test, y_pred)
print(f"KNN Model Doğruluk Oranı (Accuracy): %{acc * 100:.2f}\n")

# Sınıflandırma Raporu
print(classification_report(y_test, y_pred, target_names=['Kadın (0)', 'Erkek (1)']))

# Karmaşıklık Matrisi Görselleştirmesi
cm = confusion_matrix(y_test, y_pred)
plt.figure(figsize=(5, 4))
sns.heatmap(cm, annot=True, fmt='d', cmap='Greens', xticklabels=['Kadın (0)', 'Erkek (1)'], yticklabels=['Kadın (0)', 'Erkek (1)'])
plt.xlabel('Tahmin Edilen Sınıf')
plt.ylabel('Gerçek Sınıf')
plt.title('KNN Karmaşıklık Matrisi (k=5)')
plt.show()

### ÇÖZÜM 5: Yeni Bireyler İçin Cinsiyet Tahmini Yapma

In [ ]:
yeni_ornekler = pd.DataFrame([
    {'Kilo(kg)': 78, 'Boy(metre)': 1.76, 'Yaş': 30, 'Ayakkabı Numarası': 43},
    {'Kilo(kg)': 56, 'Boy(metre)': 1.63, 'Yaş': 25, 'Ayakkabı Numarası': 37}
])

# Yeni örnekleri scaler ile dönüştürme
yeni_ornekler_scaled = scaler.transform(yeni_ornekler)

# Model ile tahmin yapma
tahminler = model.predict(yeni_ornekler_scaled)

for i, row in yeni_ornekler.iterrows():
    etiket = 'Erkek (E)' if tahminler[i] == 1 else 'Kadın (K)'
    print(f"Birey {i+1} (Boy: {row['Boy(metre)']}m, Kilo: {row['Kilo(kg)']}kg, Ayakkabı No: {row['Ayakkabı Numarası']}) -> Tahmin: {etiket}")